# Lab 10 (capstone): Interpreting an ML portfolio

<figure>
<a
href="https://colab.research.google.com/github/quinfer/fin510-colab-notebooks/blob/main/labs/lab10_interpreting_ml.ipynb"><img
src="https://colab.research.google.com/assets/colab-badge.svg" /></a>
<figcaption>Open in Colab</figcaption>
</figure>

## Objective

A fund manager can see the weights in a machine-learning portfolio and
still want a simpler explanation of which characteristics drive its
positions. Fit that explanation as a second model. Use it to build a
portfolio and check whether it reproduces the original holdings, returns
and risk on new months.

The capstone simulates stock returns with a known interaction. Compare
an explanation using five raw characteristics with one that also
includes that interaction. **Allocation fidelity** measures how closely
the fitted weights match the original weights. **Economic fidelity**
asks how closely the two books match in return and risk.

Allow **60–90 minutes**.

After the lab, you should be able to:

-   Distinguish allocation fidelity from economic fidelity
-   Compute weight R² as the allocation-fidelity number
-   Compute the Sharpe gap and residual alpha as economic-fidelity
    numbers
-   Score an explanation on months it was not fitted on
-   Explain why a linear characteristic story can miss an interaction
    the model found

> **Assessment boundary**
>
> Use this capstone as an extension after the core backtesting material.
> It is not on Class Test 3. Follow the Blackboard brief for Coursework
> 2. The factor replication option uses HAC (Newey–West) standard errors
> for time-series regressions. The market prediction option uses
> expanding-window walk-forward evaluation.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge

rng = np.random.default_rng(42)

## Part 1: A cross-section with a known signal

Simulate 500 stocks for each of 300 months. Read one row as one stock in
one month. Its five characteristics are available at the start of that
month, and its return `r` is realised at the end. Draw each
characteristic as a standardised value with population mean zero and
standard deviation one. For example, `book_to_market` above zero means
an above-average book-to-market ratio.

Momentum contributes to expected return only when book-to-market is
above zero. Profitability contributes positively and volatility
negatively. Each month all stocks share one random size shock, scaled by
their log size. This shock has mean zero across simulations. Size then
moves realised returns around without adding to their expected return.

In [ ]:
T = 300
N = 500
features = ['momentum', 'book_to_market', 'log_size', 'volatility', 'profitability']

months = np.repeat(np.arange(T), N)
X = pd.DataFrame({f: rng.normal(0, 1, T * N) for f in features})
X['month'] = months

interaction = X['momentum'] * (X['book_to_market'] > 0).astype(float)
expected_r = 0.0125 * interaction + 0.0075 * X['profitability'] - 0.005 * X['volatility']

size_factor = rng.normal(0, 0.0125, T)
r = expected_r + size_factor[months] * X['log_size'] + rng.normal(0, 0.125, T * N)

For two simulated stocks with momentum 1, the interaction adds 1.25
percentage points to expected monthly return when book-to-market is
above average, and zero when it is at or below average. An additive
model in the five characteristics gives momentum one slope across both
groups. Part 5 introduces the product column that lets the slope vary by
group.

The simulation draws months independently. With market data, check the
dates when each characteristic and return became available.

## Part 2: Fit the flexible model and form the portfolio

Fit the boosted tree using months 0–179. At the start of each of months
180–299, use the fitted tree and that month’s characteristics to
forecast stock returns. To model a desk that favours larger firms, add
0.0175 to the score per unit of log size. That is an assumed allocation
tilt. The score sets the holdings.

Within each month, subtract the average score, then divide all centred
scores by the sum of the positive scores. The long weights sum to +1 and
the short weights sum to −1. Read a weight of +0.05 as five pence of
long exposure per pound of long capital. The arguments below use returns
per unit of long exposure.

In [ ]:
train = X['month'] < 180

model = GradientBoostingRegressor(
    n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42
)
model.fit(X.loc[train, features], r[train])

pred = X[~train].copy()
pred['r'] = r[~train]
pred['rhat'] = model.predict(pred[features])

size_tilt = 0.0175
pred['score'] = pred['rhat'] + size_tilt * pred['log_size']


def long_short_weights(s):
    w = s - s.mean()
    return w / w[w > 0].sum()


pred['w'] = pred.groupby('month')['score'].transform(long_short_weights)

## Part 3: Allocation fidelity

Use ridge to predict the original weights from the five characteristics.
Centre each characteristic within its month. After the fitted model
assigns a score to each stock, apply the same monthly long-short scaling
to make the two books comparable at £1 long and £1 short.

Fit the explanation on months 180–239. Use months 240–299 to compare its
portfolio with the original. Both sets of positions are formed before
the corresponding monthly returns arrive.

In [ ]:
fit_window = pred['month'] < 240


def demean(Z):
    return Z - Z.groupby(pred['month']).transform('mean')


def explain(Z):
    """Fit a weight model and return its scaled portfolio for later months."""
    explainer = Ridge(alpha=1.0, fit_intercept=False)
    explainer.fit(Z.loc[fit_window], pred.loc[fit_window, 'w'])
    score_months = pred.loc[~fit_window, 'month']
    scores = pd.Series(explainer.predict(Z.loc[~fit_window]), index=score_months.index)
    return scores.groupby(score_months).transform(long_short_weights)


def weight_r2(w_fit):
    w = pred.loc[~fit_window, 'w']
    return 1 - ((w - w_fit) ** 2).sum() / ((w - w.mean()) ** 2).sum()


Z_linear = demean(pred[features])
w_linear = explain(Z_linear)
print(f"Allocation fidelity (weight R², linear): {weight_r2(w_linear):.3f}")

The denominator of weight $R^2$ is the sum of squared deviations of the
original held-out weights from their mean. The numerator in the error
ratio measures the squared difference between the two books’ weights. In
this seeded run, $R^2=0.927$ means the fitted weights reduce squared
error by 92.7% relative to predicting that mean. The additive
explanation can capture the linear size tilt. Part 4 checks what the
remaining weight differences do to returns.

## Part 4: Economic fidelity

For each month, multiply each portfolio’s signed weights by the
corresponding realised stock returns and sum. For example, +0.05 on a
stock that returns 2% contributes 0.10 percentage points to the original
portfolio. A fitted weight of +0.04 contributes 0.08 points to the
fitted portfolio. This is a gross comparison using the same held-out
months for both books.

Measure each book’s annualised Sharpe as its mean monthly long-short
return divided by its monthly standard deviation, multiplied by
$\sqrt{12}$. The **Sharpe gap** subtracts the fitted book’s Sharpe from
the original book’s Sharpe. Also regress the original returns on the
fitted returns. The **residual alpha** is the intercept, which measures
average original return left after matching the realised exposure to the
fitted book. Use six lags for the HAC standard errors from Week 9.
Report the intercept in annual units by multiplying it by 12.

In [ ]:
def monthly(weights):
    s = pred.loc[~fit_window]
    return (weights * s['r']).groupby(s['month']).sum()


def sharpe(x):
    return x.mean() / x.std() * np.sqrt(12)


def economics(w_fit):
    original_r = monthly(pred.loc[~fit_window, 'w'])
    fitted_r = monthly(w_fit).rename('fitted')
    spanning = sm.OLS(original_r, sm.add_constant(fitted_r)).fit(
        cov_type='HAC', cov_kwds={'maxlags': 6}
    )
    return {
        'Sharpe (original)': sharpe(original_r),
        'Sharpe (fitted)': sharpe(fitted_r),
        'Sharpe gap': sharpe(original_r) - sharpe(fitted_r),
        'Volatility (original, p.a.)': original_r.std() * np.sqrt(12),
        'Volatility (fitted, p.a.)': fitted_r.std() * np.sqrt(12),
        'Residual alpha (p.a.)': 12 * spanning.params['const'],
        'Residual alpha t (HAC)': spanning.tvalues['const'],
    }


pd.Series(economics(w_linear)).round(3)

The original and fitted books have annualised volatilities near 9.4% and
9.5%, respectively. Their Sharpe ratios are 2.367 and 2.011, a gap of
0.356. The table’s residual alpha of 0.035 means about 3.5% per year.
Its HAC $t$ statistic of 5.79 measures how many standard errors the
estimated alpha lies above zero. The linear book captures much of the
original book’s allocation and volatility while leaving a positive
residual return in this run.

## Part 5: A richer explanation

Add the interaction from Part 1 and fit the second explanation using the
same training months. Compare the two specified explanations on the same
60 held-out months, at the same long and short exposures.

In [ ]:
rich = pred[features].copy()
rich['momentum_x_bm'] = rich['momentum'] * (rich['book_to_market'] > 0).astype(float)
w_rich = explain(demean(rich))

comparison = pd.DataFrame({
    'linear': {'Weight R²': weight_r2(w_linear), **economics(w_linear)},
    'rich': {'Weight R²': weight_r2(w_rich), **economics(w_rich)},
}).round(3)
comparison

Weight $R^2$ rises from 0.927 to 0.961. The residual alpha moves from
about 3.5% to −0.5% per year, with HAC $t$ statistics of 5.79 and −1.43.
Adding the interaction represents how the simulation makes momentum pay
in one group of stocks. In this run, that change brings residual alpha
closer to zero. Any further changes motivated by these scores need fresh
test months.

## Exercises

1.  Interpret the linear weight $R^2$ of 0.927. Which weights define the
    reference sum of squares?
2.  Locate the interaction in Part 1. Explain how it changes the
    response to momentum in two stocks with different book-to-market.
3.  Compute the Sharpe gap from the two portfolios in Part 4. Explain
    what a residual alpha of 0.035 and a HAC $t$ statistic of 5.79
    measure.
4.  Compare the richer and original books’ Sharpes. Give one reason a
    fitted book might score higher in this sample.
5.  Set `size_tilt = 0` in Part 2 and rerun from there. How do the
    linear weight $R^2$ and residual alpha change? Explain the results
    in terms of what the additive model can represent.
6.  Draw the three date blocks. Which months fit the original model, fit
    the explanation and score both books? Which dates could test a
    further change inspired by these results?

> **Why this matters**
>
> Gu, Kelly, and Xiu (2020) link forecast gains in their US equity study
> to nonlinear predictor interactions. This simulation uses one known
> interaction to demonstrate why an additive weight explanation can miss
> a return source. Kozak, Nagel, and Santosh (2018) show why
> reduced-form factor model tests can fit observed returns under
> different accounts of investor beliefs. Learning which characteristics
> describe a portfolio and identifying why its returns arise are
> different research questions.

Gu, Shihao, Bryan Kelly, and Dacheng Xiu. 2020. “Empirical Asset Pricing
via Machine Learning.” *Review of Financial Studies*.
<https://doi.org/10.1093/rfs/hhaa009>.

Kozak, Serhiy, Stefan Nagel, and Shrihari Santosh. 2018. “Interpreting
Factor Models.” *Journal of Finance* 73 (3): 1183–223.
<https://doi.org/10.1111/jofi.12612>.